# Advanced Problem Set: Iterators

**Topic:** iterator exhaustion, file objects as iterators, safe multi-pass processing, one-pass streaming algorithms, `itertools`, and robust CSV pipelines.

## Best-practice goals

1. Treat file objects as **single-use iterators**.
2. Avoid accidental second passes over the same open file.
3. Use `Path`, `with open(...)`, and explicit parsing helpers.
4. Keep lazy pipelines lazy unless materialization is intentional.
5. Document whether a function consumes an iterable once or requires re-iteration.
6. Prefer one-pass algorithms for large files.


In [1]:
from __future__ import annotations

import csv
import itertools as it
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from statistics import mean
from typing import Callable, Iterable, Iterator, Optional, TypeVar, Any

T = TypeVar("T")
U = TypeVar("U")

CSV_PATH = Path("cars.csv")
CSV_PATH


WindowsPath('cars.csv')

## Setup check — validate that `cars.csv` exists

Run this cell first. It does not load the whole file; it only checks that the path exists and prints a small preview.


In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        "cars.csv was not found. Place cars.csv in the same directory as this notebook "
        "or update CSV_PATH to the correct path."
    )

print("Found:", CSV_PATH.resolve())

with CSV_PATH.open("r", encoding="utf-8", newline="") as f:
    for i, line in zip(range(5), f):
        print(f"{i + 1:>2}: {line.rstrip()}")


## Shared parsing utilities

The second row of the file contains type names, not data. The helper functions below skip that row and parse each real car row into a typed `CarRecord`.

`open_car_rows()` is intentionally a generator. Every call opens the file again, so it is safe for repeated use.


In [3]:
@dataclass(frozen=True)
class CarRecord:
    car: str
    mpg: float
    cylinders: int
    displacement: float
    horsepower: float
    weight: float
    acceleration: float
    model: int
    origin: str


def parse_float(value: str) -> float:
    """Parse numeric CSV fields. In this dataset, missing numeric values may appear as 0."""
    return float(value)


def parse_car_row(row: dict[str, str]) -> CarRecord:
    return CarRecord(
        car=row["Car"],
        mpg=parse_float(row["MPG"]),
        cylinders=int(row["Cylinders"]),
        displacement=parse_float(row["Displacement"]),
        horsepower=parse_float(row["Horsepower"]),
        weight=parse_float(row["Weight"]),
        acceleration=parse_float(row["Acceleration"]),
        model=int(row["Model"]),
        origin=row["Origin"],
    )


def open_car_rows(path: Path = CSV_PATH) -> Iterator[dict[str, str]]:
    """
    Yield CSV rows as dictionaries, skipping the second type row.

    This function opens the file each time it is called, so calling open_car_rows()
    repeatedly gives a fresh iterator over the file.
    """
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter=";")
        next(reader)  # skip type row: STRING;DOUBLE;INT;...
        yield from reader


def open_car_records(path: Path = CSV_PATH) -> Iterator[CarRecord]:
    """Yield parsed car records lazily from cars.csv."""
    for row in open_car_rows(path):
        yield parse_car_row(row)


# Smoke test
first_three = list(it.islice(open_car_records(), 3))
first_three


[CarRecord(car='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, displacement=307.0, horsepower=130.0, weight=3504.0, acceleration=12.0, model=70, origin='US'),
 CarRecord(car='Buick Skylark 320', mpg=15.0, cylinders=8, displacement=350.0, horsepower=165.0, weight=3693.0, acceleration=11.5, model=70, origin='US'),
 CarRecord(car='Plymouth Satellite', mpg=18.0, cylinders=8, displacement=318.0, horsepower=150.0, weight=3436.0, acceleration=11.0, model=70, origin='US')]

## Problem 1 — File objects are single-use iterators

An open file object is its own iterator. If one loop reads the file, a second loop over the same file object starts from the current position, not from the beginning.

**Tasks**

1. Open `cars.csv`.
2. Read and count all lines.
3. Try to count lines again using the same file object.
4. Explain why the second count is zero.
5. Fix it by reopening the file.


In [4]:
with CSV_PATH.open("r", encoding="utf-8") as f:
    first_count = sum(1 for _ in f)
    second_count = sum(1 for _ in f)

print("First count:", first_count)
print("Second count on same file object:", second_count)

with CSV_PATH.open("r", encoding="utf-8") as f:
    fixed_count = sum(1 for _ in f)

print("Fixed count after reopening:", fixed_count)

assert first_count > 0
assert second_count == 0
assert fixed_count == first_count


First count: 408
Second count on same file object: 0
Fixed count after reopening: 408


## Problem 2 — The classic `min()` then `max()` bug on one iterator

The code below creates one iterator of MPG values and then tries to use it twice.

**Tasks**

1. Reproduce the bug.
2. Write `min_max_one_pass(values)`.
3. Use it on MPG values from `cars.csv`, ignoring `MPG == 0`.


In [5]:
def valid_mpg_values(path: Path = CSV_PATH) -> Iterator[float]:
    for record in open_car_records(path):
        if record.mpg > 0:
            yield record.mpg


mpgs = valid_mpg_values()

try:
    bad_answer = (min(mpgs), max(mpgs))
    print("Bad answer:", bad_answer)
except ValueError as exc:
    print("Expected failure:", exc)


def min_max_one_pass(values: Iterable[T]) -> tuple[T, T]:
    iterator = iter(values)
    try:
        first = next(iterator)
    except StopIteration as exc:
        raise ValueError("empty iterable") from exc

    smallest = largest = first
    for value in iterator:
        if value < smallest:
            smallest = value
        if value > largest:
            largest = value
    return smallest, largest


answer = min_max_one_pass(valid_mpg_values())
print("MPG min/max:", answer)

assert answer[0] > 0
assert answer[1] >= answer[0]


Expected failure: max() iterable argument is empty
MPG min/max: (9.0, 46.6)


## Problem 3 — Build a reusable table object

A function such as `open_car_records()` is safe because each call creates a new iterator. Another clean design is a re-iterable class.

**Tasks**

1. Implement `CarTable`.
2. Ensure each call to `iter(table)` opens the file again.
3. Show that `min`, `max`, and `sum` can be computed independently from the same table object.


In [6]:
class CarTable:
    """A re-iterable view over cars.csv."""

    def __init__(self, path: Path) -> None:
        self.path = path

    def __iter__(self) -> Iterator[CarRecord]:
        yield from open_car_records(self.path)


table = CarTable(CSV_PATH)

min_mpg = min(record.mpg for record in table if record.mpg > 0)
max_mpg = max(record.mpg for record in table if record.mpg > 0)
count_records = sum(1 for _ in table)

print("min MPG:", min_mpg)
print("max MPG:", max_mpg)
print("records:", count_records)

assert min_mpg > 0
assert max_mpg >= min_mpg
assert count_records > 0


min MPG: 9.0
max MPG: 46.6
records: 406


## Problem 4 — Normalize MPG values using a safe two-pass design

To calculate each car's MPG as a percentage of the maximum MPG, you need the maximum before producing percentages.

**Tasks**

1. Implement `mpg_percentages(path)`.
2. Reopen the file for the second pass instead of reusing the same iterator.
3. Ignore rows where `MPG == 0`.
4. Return `(car, mpg, percent_of_max)` tuples.


In [7]:
def mpg_percentages(path: Path = CSV_PATH) -> Iterator[tuple[str, float, float]]:
    """
    Yield each valid car's MPG percentage relative to the maximum valid MPG.

    This intentionally opens the file twice:
    - pass 1 finds the maximum MPG,
    - pass 2 yields normalized rows.

    This avoids materializing the whole file while still supporting a two-pass algorithm.
    """
    max_mpg = max(record.mpg for record in open_car_records(path) if record.mpg > 0)

    for record in open_car_records(path):
        if record.mpg > 0:
            yield record.car, record.mpg, record.mpg / max_mpg * 100


for car, mpg, pct in it.islice(mpg_percentages(), 10):
    print(f"{car:35s} mpg={mpg:5.1f} pct={pct:6.2f}%")

all_percentages = list(mpg_percentages())
assert all(percent > 0 for _, _, percent in all_percentages)
assert max(percent for _, _, percent in all_percentages) == 100


Chevrolet Chevelle Malibu           mpg= 18.0 pct= 38.63%
Buick Skylark 320                   mpg= 15.0 pct= 32.19%
Plymouth Satellite                  mpg= 18.0 pct= 38.63%
AMC Rebel SST                       mpg= 16.0 pct= 34.33%
Ford Torino                         mpg= 17.0 pct= 36.48%
Ford Galaxie 500                    mpg= 15.0 pct= 32.19%
Chevrolet Impala                    mpg= 14.0 pct= 30.04%
Plymouth Fury iii                   mpg= 14.0 pct= 30.04%
Pontiac Catalina                    mpg= 14.0 pct= 30.04%
AMC Ambassador DPL                  mpg= 15.0 pct= 32.19%


## Problem 5 — One-pass summary by origin

Repeatedly filtering the file for each origin is simple but may scan the file many times. A better approach is a one-pass accumulator.

**Tasks**

1. Implement `origin_mpg_summary(path)`.
2. In one pass, compute count, min, max, and average MPG by origin.
3. Ignore `MPG == 0`.


In [8]:
def origin_mpg_summary(path: Path = CSV_PATH) -> dict[str, dict[str, float]]:
    buckets: dict[str, dict[str, float]] = {}

    for record in open_car_records(path):
        if record.mpg <= 0:
            continue

        bucket = buckets.setdefault(
            record.origin,
            {"count": 0, "total": 0.0, "min": record.mpg, "max": record.mpg},
        )
        bucket["count"] += 1
        bucket["total"] += record.mpg
        bucket["min"] = min(bucket["min"], record.mpg)
        bucket["max"] = max(bucket["max"], record.mpg)

    for origin, bucket in buckets.items():
        bucket["mean"] = bucket["total"] / bucket["count"]
        del bucket["total"]

    return buckets


summary = origin_mpg_summary()
for origin, stats in sorted(summary.items()):
    print(origin, stats)

assert summary
assert all(stats["count"] > 0 for stats in summary.values())


Europe {'count': 70, 'min': 16.2, 'max': 44.3, 'mean': 27.891428571428573}
Japan {'count': 79, 'min': 18.0, 'max': 46.6, 'mean': 30.450632911392397}
US {'count': 249, 'min': 9.0, 'max': 39.0, 'mean': 20.083534136546177}


## Problem 6 — Streaming top-N with `heapq`

Using `sorted(records)` materializes the entire file. For large files, use `heapq.nlargest`.

**Tasks**

1. Implement `top_n_by_mpg(path, n)`.
2. Use a lazy generator from the file.
3. Return the top `n` valid records by MPG.


In [9]:
import heapq


def top_n_by_mpg(path: Path = CSV_PATH, n: int = 10) -> list[CarRecord]:
    if n < 0:
        raise ValueError("n must be non-negative")
    valid_records = (record for record in open_car_records(path) if record.mpg > 0)
    return heapq.nlargest(n, valid_records, key=lambda record: record.mpg)


top_10 = top_n_by_mpg(n=10)
for record in top_10:
    print(f"{record.car:35s} {record.mpg:5.1f} {record.origin}")

assert len(top_10) == 10
assert all(top_10[i].mpg >= top_10[i + 1].mpg for i in range(len(top_10) - 1))


Mazda GLC                            46.6 Japan
Honda Civic 1500 gl                  44.6 Japan
Volkswagen Rabbit C (Diesel)         44.3 Europe
Volkswagen Pickup                    44.0 Europe
Volkswagen Dasher (diesel)           43.4 Europe
Volkswagen Rabbit Custom Diesel      43.1 Europe
Volkswagen Rabbit                    41.5 Europe
Renault Lecar Deluxe                 40.9 Europe
Datsun 210                           40.8 Japan
Datsun B210 GX                       39.4 Japan


## Problem 7 — `itertools.groupby` and the shared-group iterator trap

`groupby()` returns group iterators that share the same underlying stream. If you save those group iterators and consume them later, they may be empty.

**Tasks**

1. Sort records by origin.
2. Demonstrate the bad pattern.
3. Fix it by summarizing each group immediately.


In [10]:
records_sorted_by_origin = sorted(open_car_records(), key=lambda r: r.origin)

# Bad: saving group iterators for later.
saved = [(origin, group) for origin, group in it.groupby(records_sorted_by_origin, key=lambda r: r.origin)]
bad_sizes = [(origin, len(list(group))) for origin, group in saved]
print("Bad delayed group sizes:", bad_sizes)


def groupby_origin_summary(path: Path = CSV_PATH) -> dict[str, dict[str, float]]:
    records = sorted(open_car_records(path), key=lambda r: r.origin)
    result: dict[str, dict[str, float]] = {}

    for origin, group in it.groupby(records, key=lambda r: r.origin):
        mpgs = [record.mpg for record in group if record.mpg > 0]
        result[origin] = {
            "count": len(mpgs),
            "min": min(mpgs),
            "max": max(mpgs),
            "mean": mean(mpgs),
        }

    return result


good = groupby_origin_summary()
print("Good summary:")
for origin, stats in sorted(good.items()):
    print(origin, stats)

assert good


Bad delayed group sizes: [('Europe', 0), ('Japan', 0), ('US', 0)]
Good summary:
Europe {'count': 70, 'min': 16.2, 'max': 44.3, 'mean': 27.89142857142857}
Japan {'count': 79, 'min': 18.0, 'max': 46.6, 'mean': 30.450632911392404}
US {'count': 249, 'min': 9.0, 'max': 39.0, 'mean': 20.083534136546184}


## Problem 8 — Peek at the first parsed record without losing it

Calling `next(iterator)` consumes one item. Sometimes you need to inspect the first item and still process the full stream.

**Tasks**

1. Implement `peek(iterable, default=None)`.
2. Return `(first_item, rebuilt_iterator)`.
3. Confirm that the first car still appears in the rebuilt stream.


In [11]:
def peek(iterable: Iterable[T], default: U | None = None) -> tuple[T | U | None, Iterator[T]]:
    iterator = iter(iterable)
    try:
        first = next(iterator)
    except StopIteration:
        return default, iter(())
    return first, it.chain([first], iterator)


first, rebuilt = peek(open_car_records())
print("Peeked:", first)
first_five_names = [record.car for record in it.islice(rebuilt, 5)]
print(first_five_names)

assert first is not None
assert first_five_names[0] == first.car


Peeked: CarRecord(car='Chevrolet Chevelle Malibu', mpg=18.0, cylinders=8, displacement=307.0, horsepower=130.0, weight=3504.0, acceleration=12.0, model=70, origin='US')
['Chevrolet Chevelle Malibu', 'Buick Skylark 320', 'Plymouth Satellite', 'AMC Rebel SST', 'Ford Torino']


## Problem 9 — Detect whether an object is an iterator

Many bugs start when a function assumes that an `Iterable` can be used twice. Some iterables are actually iterators.

**Tasks**

1. Implement `is_iterator(obj)`.
2. Test it on a list, generator, open file object, and `CarTable`.
3. Explain which objects are single-use.


In [12]:
def is_iterator(obj: Any) -> bool:
    try:
        return iter(obj) is obj
    except TypeError:
        return False


items = [1, 2, 3]
generator = (x * x for x in items)
table = CarTable(CSV_PATH)

print("list:", is_iterator(items))
print("generator:", is_iterator(generator))
print("CarTable:", is_iterator(table))

with CSV_PATH.open("r", encoding="utf-8") as f:
    print("open file object:", is_iterator(f))

assert is_iterator(items) is False
assert is_iterator(table) is False
assert is_iterator(generator) is True


list: False
generator: True
CarTable: False
open file object: True


## Problem 10 — Create a robust `consume_once` decorator for debugging

During development, you may want to catch accidental reuse of an iterator-producing object.

Here we create a wrapper that raises an error if a wrapped iterator is iterated after it has already been exhausted once.

**Tasks**

1. Implement `SingleUseGuard`.
2. Wrap `open_car_records()`.
3. Show that a second pass fails loudly instead of silently producing wrong results.


In [13]:
class SingleUseGuard(Iterator[T]):
    def __init__(self, iterable: Iterable[T], name: str = "iterator") -> None:
        self._iterator = iter(iterable)
        self.name = name
        self._started = False
        self._exhausted = False

    def __iter__(self) -> "SingleUseGuard[T]":
        if self._exhausted:
            raise RuntimeError(f"{self.name} has already been exhausted")
        self._started = True
        return self

    def __next__(self) -> T:
        if self._exhausted:
            raise StopIteration
        try:
            return next(self._iterator)
        except StopIteration:
            self._exhausted = True
            raise


guarded = SingleUseGuard(open_car_records(), name="car stream")
count = sum(1 for _ in guarded)
print("count:", count)

try:
    again = sum(1 for _ in guarded)
except RuntimeError as exc:
    print("Caught:", exc)

assert count > 0


count: 406
Caught: car stream has already been exhausted


## Problem 11 — Streaming validation report

Build a quality report without loading the whole file.

**Tasks**

1. Count total records.
2. Count missing-like numeric values where `MPG == 0` or `Horsepower == 0`.
3. Count records by origin.
4. Track the heaviest car.
5. Track the best MPG car.


In [14]:
def validation_report(path: Path = CSV_PATH) -> dict[str, Any]:
    total = 0
    zero_mpg = 0
    zero_horsepower = 0
    by_origin: Counter[str] = Counter()
    heaviest: CarRecord | None = None
    best_mpg: CarRecord | None = None

    for record in open_car_records(path):
        total += 1
        by_origin[record.origin] += 1

        if record.mpg == 0:
            zero_mpg += 1
        if record.horsepower == 0:
            zero_horsepower += 1

        if heaviest is None or record.weight > heaviest.weight:
            heaviest = record

        if record.mpg > 0 and (best_mpg is None or record.mpg > best_mpg.mpg):
            best_mpg = record

    return {
        "total_records": total,
        "zero_mpg": zero_mpg,
        "zero_horsepower": zero_horsepower,
        "records_by_origin": dict(by_origin),
        "heaviest_car": heaviest,
        "best_mpg_car": best_mpg,
    }


report = validation_report()
for key, value in report.items():
    print(f"{key}: {value}")

assert report["total_records"] > 0
assert report["best_mpg_car"] is not None


total_records: 406
zero_mpg: 8
zero_horsepower: 6
records_by_origin: {'US': 254, 'Europe': 73, 'Japan': 79}
heaviest_car: CarRecord(car='Pontiac Safari (sw)', mpg=13.0, cylinders=8, displacement=400.0, horsepower=175.0, weight=5140.0, acceleration=12.0, model=71, origin='US')
best_mpg_car: CarRecord(car='Mazda GLC', mpg=46.6, cylinders=4, displacement=86.0, horsepower=65.0, weight=2110.0, acceleration=17.9, model=80, origin='Japan')


## Problem 12 — Rolling window over a file iterator

A rolling average should not materialize the whole file.

**Tasks**

1. Implement `rolling_mpg(path, window_size)`.
2. Ignore rows with `MPG == 0`.
3. Yield `(car, mpg, rolling_average)` once the first full window is available.
4. Use `collections.deque(maxlen=window_size)`.


In [15]:
def rolling_mpg(path: Path = CSV_PATH, window_size: int = 5) -> Iterator[tuple[str, float, float]]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    window: deque[float] = deque(maxlen=window_size)

    for record in open_car_records(path):
        if record.mpg <= 0:
            continue

        window.append(record.mpg)
        if len(window) == window_size:
            yield record.car, record.mpg, sum(window) / window_size


for car, mpg, avg in it.islice(rolling_mpg(window_size=5), 10):
    print(f"{car:35s} mpg={mpg:5.1f} rolling_avg={avg:6.2f}")

assert len(list(it.islice(rolling_mpg(window_size=5), 3))) == 3


Ford Torino                         mpg= 17.0 rolling_avg= 16.80
Ford Galaxie 500                    mpg= 15.0 rolling_avg= 16.20
Chevrolet Impala                    mpg= 14.0 rolling_avg= 16.00
Plymouth Fury iii                   mpg= 14.0 rolling_avg= 15.20
Pontiac Catalina                    mpg= 14.0 rolling_avg= 14.80
AMC Ambassador DPL                  mpg= 15.0 rolling_avg= 14.40
Dodge Challenger SE                 mpg= 15.0 rolling_avg= 14.40
Plymouth 'Cuda 340                  mpg= 14.0 rolling_avg= 14.40
Chevrolet Monte Carlo               mpg= 15.0 rolling_avg= 14.60
Buick Estate Wagon (sw)             mpg= 14.0 rolling_avg= 14.60


## Problem 13 — Compare materialization, reopening, and one-pass strategies

Implement three functions that return the same answer: min MPG, max MPG, and count of valid MPG rows.

**Tasks**

1. `summary_materialized(path)` loads valid MPG values into a list.
2. `summary_reopen(path)` opens the file separately for each aggregation.
3. `summary_one_pass(path)` computes all values in one pass.
4. Verify that all three answers match.


In [16]:
def summary_materialized(path: Path = CSV_PATH) -> dict[str, float]:
    values = list(valid_mpg_values(path))
    return {"count": len(values), "min": min(values), "max": max(values)}


def summary_reopen(path: Path = CSV_PATH) -> dict[str, float]:
    return {
        "count": sum(1 for _ in valid_mpg_values(path)),
        "min": min(valid_mpg_values(path)),
        "max": max(valid_mpg_values(path)),
    }


def summary_one_pass(path: Path = CSV_PATH) -> dict[str, float]:
    count = 0
    smallest = None
    largest = None

    for mpg in valid_mpg_values(path):
        count += 1
        smallest = mpg if smallest is None else min(smallest, mpg)
        largest = mpg if largest is None else max(largest, mpg)

    if count == 0:
        raise ValueError("no valid MPG values")

    return {"count": count, "min": smallest, "max": largest}


a = summary_materialized()
b = summary_reopen()
c = summary_one_pass()

print(a)
print(b)
print(c)

assert a == b == c


{'count': 398, 'min': 9.0, 'max': 46.6}
{'count': 398, 'min': 9.0, 'max': 46.6}
{'count': 398, 'min': 9.0, 'max': 46.6}


## Problem 14 — Build reusable filtering functions

Good iterator code often uses small generator functions. Implement filters that can be composed without loading the file.

**Tasks**

1. `where_origin(records, origin)`.
2. `where_model_between(records, start, end)`.
3. `where_valid_mpg(records)`.
4. Compose them to find the top 5 Japanese cars from model years 1978 to 1982.


In [17]:
def where_origin(records: Iterable[CarRecord], origin: str) -> Iterator[CarRecord]:
    for record in records:
        if record.origin == origin:
            yield record


def where_model_between(records: Iterable[CarRecord], start: int, end: int) -> Iterator[CarRecord]:
    for record in records:
        if start <= record.model <= end:
            yield record


def where_valid_mpg(records: Iterable[CarRecord]) -> Iterator[CarRecord]:
    for record in records:
        if record.mpg > 0:
            yield record


pipeline = where_valid_mpg(
    where_model_between(
        where_origin(open_car_records(), "Japan"),
        78,
        82,
    )
)

top = heapq.nlargest(5, pipeline, key=lambda record: record.mpg)

for record in top:
    print(record)

assert len(top) <= 5
assert all(record.origin == "Japan" and 78 <= record.model <= 82 and record.mpg > 0 for record in top)


CarRecord(car='Mazda GLC', mpg=46.6, cylinders=4, displacement=86.0, horsepower=65.0, weight=2110.0, acceleration=17.9, model=80, origin='Japan')
CarRecord(car='Honda Civic 1500 gl', mpg=44.6, cylinders=4, displacement=91.0, horsepower=67.0, weight=1850.0, acceleration=13.8, model=80, origin='Japan')
CarRecord(car='Datsun 210', mpg=40.8, cylinders=4, displacement=85.0, horsepower=65.0, weight=2110.0, acceleration=19.2, model=80, origin='Japan')
CarRecord(car='Datsun B210 GX', mpg=39.4, cylinders=4, displacement=85.0, horsepower=70.0, weight=2070.0, acceleration=18.6, model=78, origin='Japan')
CarRecord(car='Toyota Starlet', mpg=39.1, cylinders=4, displacement=79.0, horsepower=58.0, weight=1755.0, acceleration=16.9, model=81, origin='Japan')


## Problem 15 — Final challenge: streaming CSV profiler

Write `profile_cars_csv(path)` that scans the file once and returns a rich profile.

**Requirements**

Return a dictionary with:

- total record count,
- count by origin,
- count by cylinder value,
- min/max/mean MPG excluding zero MPG,
- min/max/mean weight,
- top 3 MPG cars,
- bottom 3 valid MPG cars,
- number of rows with zero MPG,
- number of rows with zero horsepower.

Use one pass over `open_car_records(path)`. For top and bottom cars, keeping small lists with `heapq` is allowed.


In [18]:
def profile_cars_csv(path: Path = CSV_PATH) -> dict[str, Any]:
    total = 0
    origin_counts: Counter[str] = Counter()
    cylinder_counts: Counter[int] = Counter()

    zero_mpg = 0
    zero_horsepower = 0

    mpg_count = 0
    mpg_total = 0.0
    mpg_min: float | None = None
    mpg_max: float | None = None

    weight_count = 0
    weight_total = 0.0
    weight_min: float | None = None
    weight_max: float | None = None

    top_mpg: list[tuple[float, int, CarRecord]] = []
    bottom_mpg: list[tuple[float, int, CarRecord]] = []
    sequence = 0

    for record in open_car_records(path):
        total += 1
        sequence += 1
        origin_counts[record.origin] += 1
        cylinder_counts[record.cylinders] += 1

        if record.mpg == 0:
            zero_mpg += 1
        else:
            mpg_count += 1
            mpg_total += record.mpg
            mpg_min = record.mpg if mpg_min is None else min(mpg_min, record.mpg)
            mpg_max = record.mpg if mpg_max is None else max(mpg_max, record.mpg)

            # Keep top 3 MPG cars using a min-heap.
            heapq.heappush(top_mpg, (record.mpg, sequence, record))
            if len(top_mpg) > 3:
                heapq.heappop(top_mpg)

            # Keep bottom 3 valid MPG cars using a max-heap simulated with negative MPG.
            heapq.heappush(bottom_mpg, (-record.mpg, sequence, record))
            if len(bottom_mpg) > 3:
                heapq.heappop(bottom_mpg)

        if record.horsepower == 0:
            zero_horsepower += 1

        weight_count += 1
        weight_total += record.weight
        weight_min = record.weight if weight_min is None else min(weight_min, record.weight)
        weight_max = record.weight if weight_max is None else max(weight_max, record.weight)

    if total == 0:
        raise ValueError("cars.csv contains no data rows")

    top_mpg_records = [item[2] for item in sorted(top_mpg, key=lambda x: x[0], reverse=True)]
    bottom_mpg_records = [item[2] for item in sorted(bottom_mpg, key=lambda x: -x[0])]

    return {
        "total_records": total,
        "origin_counts": dict(origin_counts),
        "cylinder_counts": dict(cylinder_counts),
        "mpg": {
            "count": mpg_count,
            "min": mpg_min,
            "max": mpg_max,
            "mean": mpg_total / mpg_count if mpg_count else None,
        },
        "weight": {
            "count": weight_count,
            "min": weight_min,
            "max": weight_max,
            "mean": weight_total / weight_count if weight_count else None,
        },
        "top_3_mpg_cars": top_mpg_records,
        "bottom_3_valid_mpg_cars": bottom_mpg_records,
        "zero_mpg": zero_mpg,
        "zero_horsepower": zero_horsepower,
    }


profile = profile_cars_csv()

print("Total records:", profile["total_records"])
print("Origin counts:", profile["origin_counts"])
print("Cylinder counts:", profile["cylinder_counts"])
print("MPG stats:", profile["mpg"])
print("Weight stats:", profile["weight"])
print("\nTop 3 MPG cars:")
for record in profile["top_3_mpg_cars"]:
    print(f"  {record.car:35s} {record.mpg:5.1f}")

print("\nBottom 3 valid MPG cars:")
for record in profile["bottom_3_valid_mpg_cars"]:
    print(f"  {record.car:35s} {record.mpg:5.1f}")

assert profile["total_records"] > 0
assert profile["mpg"]["max"] >= profile["mpg"]["min"]
assert len(profile["top_3_mpg_cars"]) <= 3
assert len(profile["bottom_3_valid_mpg_cars"]) <= 3


Total records: 406
Origin counts: {'US': 254, 'Europe': 73, 'Japan': 79}
Cylinder counts: {8: 108, 4: 207, 6: 84, 3: 4, 5: 3}
MPG stats: {'count': 398, 'min': 9.0, 'max': 46.6, 'mean': 23.514572864321615}
Weight stats: {'count': 406, 'min': 1613.0, 'max': 5140.0, 'mean': 2979.4137931034484}

Top 3 MPG cars:
  Mazda GLC                            46.6
  Honda Civic 1500 gl                  44.6
  Volkswagen Rabbit C (Diesel)         44.3

Bottom 3 valid MPG cars:
  Hi 1200D                              9.0
  Ford F250                            10.0
  Chevy C20                            10.0


## Extra practice prompts

Use these as follow-up exercises:

1. Add a function that writes `mpg_percentages()` to `cars_mpg_percentages.csv`.
2. Create a generator that yields only cars whose horsepower-to-weight ratio is above the dataset average.
3. Implement a one-pass correlation-like summary between `weight` and `mpg`.
4. Add a validation rule that reports rows where `horsepower == 0` but `mpg > 0`.
5. Rewrite `profile_cars_csv()` so it can profile any semicolon-delimited file using a schema dictionary.
6. Benchmark `summary_materialized`, `summary_reopen`, and `summary_one_pass` with `timeit`.
7. Modify `CarTable` to accept any delimiter.
8. Add unit tests with `pytest` for `parse_car_row`, `valid_mpg_values`, and `profile_cars_csv`.


## Takeaways

- `open("cars.csv")` returns a single-use iterator over lines.
- Calling `min()` on a generator consumes it.
- Reopen files for clean repeated passes, or design a re-iterable class such as `CarTable`.
- For large files, prefer one-pass accumulators and streaming algorithms.
- Materialization is acceptable when it is intentional, documented, and bounded.
- `itertools.groupby` group iterators must be consumed immediately.
- Small generator filters make data pipelines readable and memory efficient.
